# Reporte Técnico: Ingeniería de Variables y Modelado de Tablas Analíticas Base (ABT)
**Área de Desarrollo:** Ingeniería de Datos / Analytics Engineering  
**Pipeline de Origen:** `3_src/feature_engineering.py`  
**Objetivo Estratégico:** Preparación de datos para el análisis del consumidor - Mundial FIFA 2026  

---

## 1. Introducción y Propósito del Módulo
El propósito de este cuaderno de trabajo es documentar de forma detallada la fase de **Ingeniería de Variables (Feature Engineering)**. Esta etapa se encarga de recibir el conjunto de datos que ya ha pasado por los controles de calidad iniciales (`dataset_limpio.csv`) y transformarlo en estructuras optimizadas llamadas **Analytical Base Tables (ABT)**.

El objetivo central del trabajo realizado es descentralizar el procesamiento computacional. Al resolver la complejidad del modelado directamente en la tubería de datos con Python, se garantiza que la capa posterior de Inteligencia de Negocios y visualización de tableros consuma estructuras optimizadas, eliminando retrasos en las consultas y asegurando la consistencia en los indicadores clave de rendimiento (KPIs).

## 2. Diagnóstico del Set de Datos y Soluciones de Ingeniería Implementadas

Para dar respuesta exacta a los requerimientos de la estrategia de marketing y estructurar las métricas solicitadas, el script principal ejecuta tres transformaciones clave de manera secuencial:

### Fase A: Estandarización de Variables Categóricas (Remoción de Ruido)
* **Descripción del problema:** En las respuestas de texto semiestructuradas, con frecuencia existen inconsistencias debido a espacios en blanco accidentales al inicio o al final de las palabras (por ejemplo, diferencias entre `" salado "` y `"salado"`). Para un motor de base de datos o de visualización, estas variaciones se interpretan como categorías separadas, lo que fragmenta y distorsiona el cálculo real de los totales.
* **Solución aplicada:** Se implementó una limpieza sistemática que recorre las columnas categóricas críticas (tales como `SaborPreferido` y `GastoSnacksPartido`), aplicando las funciones vectorizadas `.str.strip()` para eliminar caracteres vacíos y `.str.title()` para homologar la capitalización del texto.

### Fase B: Construcción de Indicadores Condicionales Avanzados
Para enriquecer el análisis del consumidor sin sobrecargar la capa de visualización con fórmulas complejas en tiempo de ejecución, se calculó la lógica condicional directamente en el pipeline mediante la función estructurada `np.select()` de NumPy:
1.  **`Disposicion_Gasto_Premium`**: Segmenta el perfil económico del encuestado cruzando el presupuesto declarado por partido con la intención de adquirir productos especiales de edición mundialista. Clasifica a la muestra en perfiles de gasto alto (*Premium Alta*), intermedio (*Moderada*) o restringido (*Sensible Al Precio*).
2.  **`Segmento_Lealtad_Mundial`**: Evalúa de manera integral el nivel de compromiso comercial del consumidor. Mide la coincidencia simultánea entre la intención de visualizar el torneo, el interés por la indumentaria oficial y la disposición hacia el coleccionismo. Clasifica los registros en: *Fanático Target (Alto)*, *Casual* o *Espectador Pasivo*.
3.  **`Segmento_Edad_Analitico`**: Simplifica la dimensionalidad de los rangos de edad originales, agrupándolos en cuatro grandes clústeres demográficos que facilitan el filtrado ejecutivo en los informes de alta gerencia.

### Fase C: Normalización Estructural mediante Desanidamiento Multidimensional
* **Descripción del problema:** Las preguntas relacionadas con la selección de marcas, snacks y líderes de opinión permitían múltiples opciones en una sola celda, separadas por puntos y comas (`;`). Mantener los datos en este formato impide calcular la verdadera participación de mercado de forma individual, ya que las cadenas de texto quedan agrupadas de forma masiva.
* **Solución aplicada:** Se transformaron las cadenas de texto en colecciones mediante el método `.str.split(';')` y posteriormente se utilizó la función **`.explode()`** de Pandas. Este proceso expande las respuestas de selección múltiple en registros independientes distribuidos en nuevas filas, vinculados siempre al identificador único de la encuesta (`EncuestaID`). Esto permite un conteo preciso y limpio de cada opción por separado para las gráficas de distribución.

# =========================================================================
# COMPONENTE DE AUDITORÍA Y ORQUESTACIÓN DEL PIPELINE DE DATOS
# =========================================================================
import os
import sys

# Inclusión de la ruta raíz para la localización del módulo de código fuente
sys.path.append(os.path.abspath("../"))
from _3_src.feature_engineering import AnalyticsEngineerPipeline

# Definición de rutas del entorno local de desarrollo
RUTA_INPUT_LIMPIO = "../1_data/clean/dataset_limpio.csv"
CARPETA_OUTPUT_ABT = "../1_data/clean"

print("[INFO] Iniciando el proceso de orquestación y validación de datos...\n")

try:
    # Inicialización de la tubería analítica
    pipeline = AnalyticsEngineerPipeline(RUTA_INPUT_LIMPIO, CARPETA_OUTPUT_ABT)

    # Carga del set de datos limpio de la fase anterior
    df_origen = pipeline.cargar_dataset_previo()

    # Ejecución de la limpieza analítica y la ingeniería de variables
    pipeline.refinar_strings_analiticos()
    df_maestro = pipeline.ejecutar_feature_engineering()
    df_snacks = pipeline.construir_abt_snacks()
    df_jugadores = pipeline.construir_abt_jugadores()

    # Despliegue de métricas cuantitativas para control de calidad
    print("=" * 65)
    print("📋 REPORTE DE AUDITORÍA DE INFRAESTRUCTURA DE DATOS")
    print("=" * 65)
    print(
        f"  [TABLA MAESTRA] Filas generales procesadas:    {df_maestro.shape[0]} registros."
    )
    print(
        f"  [ABT SNACKS]    Filas granulares normalizadas: {df_snacks.shape[0]} registros."
    )
    print(
        f"  [ABT JUGADORES] Filas granulares normalizadas: {df_jugadores.shape[0]} registros."
    )
    print("=" * 65)
    print(
        "🚀 ESTADO: Integración completada. Archivos ABT exportados correctamente."
    )

except Exception as error:
    print(
        f"❌ ALERTA: Se detectó una inconsistencia en la ejecución del pipeline: {error}"
    )

## 3. Recomendaciones de Integración para la Capa de Inteligencia de Negocios (BI)

Para garantizar la precisión de los reportes y evitar duplicidades en los resultados, el consumo de las tablas generadas debe regirse bajo los siguientes lineamientos de modelado:

1.  **Estructura del Modelo:** Se debe implementar un modelo con diseño de esquema en estrella. La tabla maestra `ABT_Mundial.csv` funcionará como el núcleo relacional, vinculándose a las tablas granulares de comportamiento (`ABT_Snacks.csv` y `ABT_Jugadores.csv`) mediante relaciones de uno a muchos (1:N) a través del campo común `EncuestaID`.
2.  **Análisis de Preferencias Específicas:** Las métricas de volumen, participación y frecuencia para snacks y líderes de opinión deben calcularse exclusivamente desde sus respectivas bases desanidadas (`ABT_Snacks` y `ABT_Jugadores`) para asegurar que el conteo refleje cada mención individual.
3.  **Uso de Dimensiones de Control:** Se sugiere priorizar el uso de la variable calculada `Segmento_Edad_Analitico` para los filtros demográficos principales, optimizando el rendimiento visual de los tableros de control.